# BERT Text Classification Demo

All three protocols: REST JSON, gRPC, REST Binary

**Important:** Run cells in order. REST JSON must run first to avoid tritonclient state issues.

In [ ]:
import os
import time
import numpy as np
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient
from transformers import AutoTokenizer

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"

os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else None

print(f"Namespace: {NAMESPACE}")
print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Prepare inputs
# =============================================================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "I love this product!",
    "This is terrible.",
]

print(f"Texts: {texts}")

url = REST_URL.replace("http://", "").replace("https://", "")

In [ ]:
# =============================================================================
# 1. REST (JSON) - MUST RUN FIRST
# =============================================================================
encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="np")
ids_json = encoded["input_ids"].astype(np.int64).copy()
mask_json = encoded["attention_mask"].astype(np.int64).copy()

client = httpclient.InferenceServerClient(url=url)
inp_ids = httpclient.InferInput("input_ids", list(ids_json.shape), "INT64")
inp_ids.set_data_from_numpy(ids_json, binary_data=False)
inp_mask = httpclient.InferInput("attention_mask", list(mask_json.shape), "INT64")
inp_mask.set_data_from_numpy(mask_json, binary_data=False)
out = httpclient.InferRequestedOutput("logits", binary_data=False)

start = time.time()
resp = client.infer("bert-base-uncased", [inp_ids, inp_mask], outputs=[out], headers=http_headers)
t_json = (time.time() - start) * 1000
logits_json = resp.as_numpy("logits")

preds_json = ['NEGATIVE' if l[0] > l[1] else 'POSITIVE' for l in logits_json]
print(f"✓ REST (JSON):   {t_json:8.2f} ms | Predictions: {preds_json}")

In [ ]:
# =============================================================================
# 2. gRPC Protocol
# =============================================================================
encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="np")
ids_grpc = encoded["input_ids"].astype(np.int64).copy()
mask_grpc = encoded["attention_mask"].astype(np.int64).copy()

client = grpcclient.InferenceServerClient(url=GRPC_URL)
inp_ids = grpcclient.InferInput("input_ids", list(ids_grpc.shape), "INT64")
inp_ids.set_data_from_numpy(ids_grpc)
inp_mask = grpcclient.InferInput("attention_mask", list(mask_grpc.shape), "INT64")
inp_mask.set_data_from_numpy(mask_grpc)
out = grpcclient.InferRequestedOutput("logits")

start = time.time()
resp = client.infer("bert-base-uncased", [inp_ids, inp_mask], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
logits_grpc = resp.as_numpy("logits")
client.close()

preds_grpc = ['NEGATIVE' if l[0] > l[1] else 'POSITIVE' for l in logits_grpc]
print(f"✓ gRPC:          {t_grpc:8.2f} ms | Predictions: {preds_grpc}")

In [ ]:
# =============================================================================
# 3. REST (Binary) Protocol
# =============================================================================
encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="np")
ids_binary = encoded["input_ids"].astype(np.int64).copy()
mask_binary = encoded["attention_mask"].astype(np.int64).copy()

client = httpclient.InferenceServerClient(url=url)
inp_ids = httpclient.InferInput("input_ids", list(ids_binary.shape), "INT64")
inp_ids.set_data_from_numpy(ids_binary, binary_data=True)
inp_mask = httpclient.InferInput("attention_mask", list(mask_binary.shape), "INT64")
inp_mask.set_data_from_numpy(mask_binary, binary_data=True)
out = httpclient.InferRequestedOutput("logits", binary_data=True)

start = time.time()
resp = client.infer("bert-base-uncased", [inp_ids, inp_mask], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
logits_binary = resp.as_numpy("logits")

preds_binary = ['NEGATIVE' if l[0] > l[1] else 'POSITIVE' for l in logits_binary]
print(f"✓ REST (Binary): {t_binary:8.2f} ms | Predictions: {preds_binary}")

In [ ]:
# =============================================================================
# Verify outputs match
# =============================================================================
print(f"Outputs match (JSON == gRPC):   {np.allclose(logits_json, logits_grpc)}")
print(f"Outputs match (gRPC == Binary): {np.allclose(logits_grpc, logits_binary)}")